# Multi-Tier Caching System

**Multigen SDK — Notebook 22**

---

## Overview

Caching is one of the most effective performance optimizations in AI agent pipelines.
LLM calls, embedding computations, and external API fetches are expensive and
often deterministic for the same inputs — caching them can cut latency by 10–100×.

The Multigen caching system provides four cooperating primitives:

| Class | Type | Eviction | Async | Best for |
|-------|------|----------|-------|----------|
| `LRUCache` | Synchronous | LRU + optional TTL | No | CPU-bound, sync code |
| `TTLCache` | Synchronous | TTL-only (lazy/eager) | No | Time-sensitive tokens |
| `AsyncCache` | Async | LRU + TTL + stampede guard | Yes | Async functions, I/O |
| `MultiTierCache` | Async L1 + L2 | Per-tier LRU + TTL | Yes | Hot/warm access patterns |

Two supporting utilities:

- `CacheManager` — a named registry for managing multiple `AsyncCache` instances.
- `@cached` — a decorator that wraps any async function with a built-in `AsyncCache`.
- `make_cache_key` — deterministic key generation from arbitrary Python arguments.

## What this notebook covers

| Section | Topic |
|---------|-------|
| 1 | Setup and imports |
| 2 | LRUCache — set/get, capacity eviction, hit/miss stats, per-entry TTL |
| 3 | TTLCache — expiry, eager eviction, stats |
| 4 | AsyncCache — async get_or_compute, stampede protection |
| 5 | @cached decorator — wrap async function, cache_clear |
| 6 | make_cache_key — stable key generation |
| 7 | MultiTierCache — L1 + L2, write-through, L2 backfill |
| 8 | CacheManager — named registry, clear_all |
| 9 | Cache with expensive computation — measure speedup |
| 10 | Cache invalidation patterns |
| 11 | Caching in an agent pipeline |
| 12 | Summary: when to use each tier |

No external APIs required — all I/O is simulated with `asyncio.sleep`.


## Section 1 — Setup and Imports

All cache classes are exported from the top-level `multigen` package.  We import:

- `LRUCache` — synchronous LRU with optional per-entry TTL
- `TTLCache` — synchronous TTL-only store
- `AsyncCache` — async LRU + TTL + stampede protection
- `MultiTierCache` — two-tier async cache (L1 + L2)
- `CacheManager` — named cache registry
- `cached` — async function caching decorator
- `make_cache_key` — deterministic key generator


In [ ]:
import asyncio
import os
import sys
import time

# ── SDK path resolution ────────────────────────────────────────────────────────
for candidate in ['../sdk', 'sdk']:
    candidate = os.path.abspath(candidate)
    if os.path.isdir(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)

from multigen import (
    LRUCache,
    TTLCache,
    AsyncCache,
    MultiTierCache,
    CacheManager,
    cached,
    make_cache_key,
)

print('LRUCache       :', LRUCache)
print('TTLCache       :', TTLCache)
print('AsyncCache     :', AsyncCache)
print('MultiTierCache :', MultiTierCache)
print('CacheManager   :', CacheManager)
print('cached         :', cached)
print('make_cache_key :', make_cache_key)
print()
print('All imports successful.')


## Section 2 — LRUCache

`LRUCache` is a synchronous **Least-Recently-Used** cache backed by a Python
`collections.OrderedDict`.  Key features:

- **LRU eviction** — when `maxsize` is reached the least-recently-used entry
  (the one whose last `get` was earliest) is removed to make room.
- **Optional per-entry TTL** — pass `ttl=` to `set()` to override the
  `default_ttl` set at construction time.  Expired entries are removed lazily
  on `get`.
- **Hit/miss tracking** — `stats()` returns `hits`, `misses`, and `hit_rate`.

### Constructor parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `maxsize` | 256 | Maximum number of live entries |
| `default_ttl` | `None` | Default TTL in seconds per entry; `None` = immortal |


In [ ]:
# ── Basic set / get ───────────────────────────────────────────────────────────

lru = LRUCache(maxsize=4, default_ttl=None)

lru.set('a', 'alpha')
lru.set('b', 'beta')
lru.set('c', 'gamma')
lru.set('d', 'delta')

print('After setting 4 entries (maxsize=4):')
print(f'  get("a") → {lru.get("a")!r}')
print(f'  get("b") → {lru.get("b")!r}')
print(f'  get("c") → {lru.get("c")!r}')
print(f'  get("d") → {lru.get("d")!r}')
print(f'  stats    → {lru.stats()}')
print()

# ── LRU eviction ──────────────────────────────────────────────────────────────
# "a" was accessed most recently above; "b" was second most recent.
# The current LRU order (oldest first): d was last touched, then c, b, a.
# But we just accessed a, b, c, d in sequence — so d is now MRU.
# Inserting "e" evicts the LRU entry.

lru.set('e', 'epsilon')   # triggers eviction of the LRU entry

print('After inserting 5th entry (capacity=4 → one evicted):')
print(f'  len(lru) → {len(lru)}')
print(f'  get("a") → {lru.get("a")!r}   (None means evicted)')
print(f'  get("e") → {lru.get("e")!r}')
print(f'  stats    → {lru.stats()}')
print()

# ── Per-entry TTL ─────────────────────────────────────────────────────────────
lru2 = LRUCache(maxsize=100)
lru2.set('immortal', 42)
lru2.set('expires',  99, ttl=0.05)   # expires in 50 ms

print('Per-entry TTL (ttl=0.05 s):')
print(f'  get("immortal") immediately → {lru2.get("immortal")!r}')
print(f'  get("expires")  immediately → {lru2.get("expires")!r}')

time.sleep(0.1)
print(f'  get("expires")  after 0.1 s → {lru2.get("expires")!r}  ← expired')
print(f'  get("immortal") after 0.1 s → {lru2.get("immortal")!r}  ← still alive')
print()
print(f'Final stats: {lru2.stats()}')


## Section 3 — TTLCache

`TTLCache` enforces a **time-to-live** policy as its primary eviction mechanism:

- Every entry has a TTL (defaults to `default_ttl` passed at construction).
- Expired entries are removed **lazily** on `get` and **eagerly** before every `set`
  (the `_evict_expired` sweep runs before inserting a new entry).
- When the store is full after eviction, the **oldest** entry (by `created_at`) is
  removed to make room.

`TTLCache` is ideal for ephemeral tokens (auth tokens, rate-limit buckets, OTP
codes) where expiry semantics matter more than access recency.


In [ ]:
ttlc = TTLCache(default_ttl=300.0, maxsize=1000)

# Store entries — some with custom TTL, some using the default
ttlc.set('auth_token',     'tok-abc123',  ttl=0.15)   # short-lived: 150 ms
ttlc.set('api_result',     {'rows': 42},  ttl=60.0)   # medium: 60 s
ttlc.set('static_config',  {'version': 3})            # uses default_ttl=300 s

print('Immediately after storing:')
print(f'  get("auth_token")     → {ttlc.get("auth_token")!r}')
print(f'  get("api_result")     → {ttlc.get("api_result")!r}')
print(f'  get("static_config")  → {ttlc.get("static_config")!r}')
print(f'  len(ttlc)             → {len(ttlc)}')
print()

# Wait for the short-lived token to expire
time.sleep(0.2)

print('After 0.2 s (auth_token TTL was 0.15 s):')
print(f'  get("auth_token")     → {ttlc.get("auth_token")!r}     ← expired')
print(f'  get("api_result")     → {ttlc.get("api_result")!r}  ← alive')
print(f'  len(ttlc)             → {len(ttlc)}      ← expired entries swept')
print()

# Manual invalidation
removed = ttlc.invalidate('api_result')
print(f'invalidate("api_result") → {removed}')
print(f'get("api_result") after invalidate → {ttlc.get("api_result")!r}')
print()

# TTLCache maxsize eviction: fill to capacity and observe oldest-entry eviction
small_ttlc = TTLCache(default_ttl=60.0, maxsize=3)
small_ttlc.set('x1', 'first')
small_ttlc.set('x2', 'second')
small_ttlc.set('x3', 'third')
small_ttlc.set('x4', 'fourth')   # triggers eviction of oldest entry ('x1')

print('TTLCache maxsize eviction (maxsize=3, inserting 4th):')
print(f'  get("x1") → {small_ttlc.get("x1")!r}  ← evicted as oldest')
print(f'  get("x2") → {small_ttlc.get("x2")!r}')
print(f'  get("x3") → {small_ttlc.get("x3")!r}')
print(f'  get("x4") → {small_ttlc.get("x4")!r}')


## Section 4 — AsyncCache: Stampede Protection

`AsyncCache` is the async counterpart of `LRUCache`.  Its headline feature is
**stampede protection** (also called single-flight or request coalescing):

> When multiple coroutines concurrently request the same key that is not yet
> cached, only **one** of them runs the computation.  All others wait on an
> `asyncio.Event` and receive the result when it arrives — no duplicate work.

This matters for expensive operations (embedding generation, LLM calls) that might
be triggered simultaneously by multiple agents.

### `get_or_compute` API

```python
value = await cache.get_or_compute(
    key,
    compute_fn,    # callable (sync or async)
    ttl,           # optional TTL override
    *args,         # forwarded to compute_fn
    **kwargs,
)
```

If `key` is in the cache, `compute_fn` is never called.


In [ ]:
async_cache = AsyncCache(maxsize=128, default_ttl=60.0)

# ── Basic async get / set ──────────────────────────────────────────────────────

await async_cache.set('greeting', 'Hello, World!')
val = await async_cache.get('greeting')
print(f'set + get: {val!r}')

miss = await async_cache.get('nonexistent')
print(f'get (miss): {miss!r}')
print()

# ── get_or_compute with async factory ─────────────────────────────────────────

call_count = {'n': 0}

async def expensive_embedding(text: str) -> list:
    """Simulates a slow embedding API call."""
    call_count['n'] += 1
    await asyncio.sleep(0.05)   # 50 ms "API call"
    return [hash(w) % 100 for w in text.split()[:5]]

key = 'embed:hello world'

t0 = time.perf_counter()
result1 = await async_cache.get_or_compute(key, expensive_embedding, 30.0, 'hello world')
t1 = time.perf_counter()

result2 = await async_cache.get_or_compute(key, expensive_embedding, 30.0, 'hello world')
t2 = time.perf_counter()

print(f'First call  (computed): {result1}  time={(t1-t0)*1000:.1f} ms  calls={call_count["n"]}')
print(f'Second call (cached):   {result2}  time={(t2-t1)*1000:.2f} ms  calls={call_count["n"]}')
print()

# ── Stampede protection ────────────────────────────────────────────────────────
# Launch two concurrent tasks for the same uncached key.
# Only one computation should run.

await async_cache.clear()
call_count['n'] = 0

stampede_key = 'embed:concurrent test'

async def task_a():
    return await async_cache.get_or_compute(stampede_key, expensive_embedding, 60.0, 'concurrent test')

async def task_b():
    # Start slightly after task_a to ensure it's already in-flight
    await asyncio.sleep(0.01)
    return await async_cache.get_or_compute(stampede_key, expensive_embedding, 60.0, 'concurrent test')

results = await asyncio.gather(task_a(), task_b())

print(f'Stampede test — two concurrent tasks for the same key:')
print(f'  task_a result : {results[0]}')
print(f'  task_b result : {results[1]}')
print(f'  call_count    : {call_count["n"]}  (expected 1 — stampede protection worked)')
print()
stats = await async_cache.stats()
print(f'AsyncCache stats: {stats}')


## Section 5 — @cached Decorator

`@cached` wraps any async function with an `AsyncCache` instance.  The decorator:

1. Generates a cache key from the function arguments using `make_cache_key`.
2. Returns the cached value on subsequent calls with the same arguments.
3. Attaches `wrapper.cache` (the underlying `AsyncCache`) and `wrapper.cache_clear`
   for inspection and manual invalidation.

### Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `ttl` | `None` | Time-to-live for cached results |
| `maxsize` | 128 | Maximum number of cached results |
| `key_fn` | `None` | Custom key function; defaults to `make_cache_key` |


In [ ]:
call_log = []

@cached(ttl=30.0, maxsize=256)
async def analyse_text(text: str, model: str = 'default') -> dict:
    """Simulate an expensive NLP analysis call."""
    call_log.append({'text': text, 'model': model})
    await asyncio.sleep(0.03)   # 30 ms "model inference"
    return {
        'text': text,
        'model': model,
        'word_count': len(text.split()),
        'sentiment': 'positive' if 'good' in text.lower() else 'neutral',
    }

# First call — computed
r1 = await analyse_text('This is a good result', model='bert-mock')
print(f'First call  (computed)  : {r1}')
print(f'  call_log length       : {len(call_log)}')
print()

# Second call with identical arguments — cached
r2 = await analyse_text('This is a good result', model='bert-mock')
print(f'Second call (cached)    : {r2}')
print(f'  call_log length       : {len(call_log)}  ← no new call')
print()

# Different arguments — computed fresh
r3 = await analyse_text('The weather is neutral', model='bert-mock')
print(f'Third call (new args)   : {r3}')
print(f'  call_log length       : {len(call_log)}  ← new computation')
print()

# Inspect the underlying cache
cache_stats = await analyse_text.cache.stats()
print(f'Cache stats: {cache_stats}')
print()

# Clear the cache — next call will recompute
await analyse_text.cache_clear()
call_log.clear()
r4 = await analyse_text('This is a good result', model='bert-mock')
print(f'After cache_clear — recomputed: {r4}')
print(f'  call_log length after clear : {len(call_log)}')


## Section 6 — make_cache_key

`make_cache_key(*args, **kwargs)` produces a stable **16-character hex** string
derived from a SHA-256 hash of the JSON-serialised arguments.

Properties:
- **Stable** — same arguments always produce the same key, regardless of dict
  insertion order (keys are `sort_keys=True`-serialised).
- **Collision-resistant** — 16 hex chars = 64 bits; collisions are astronomically
  unlikely for typical cache sizes.
- **Type-agnostic** — lists, dicts, nested structures, and non-serialisable objects
  (falls back to `str()`) are all handled.
- **Compact** — 16 chars is short enough to use as a dict key without bloating
  memory.


In [ ]:
# ── Basic usage ────────────────────────────────────────────────────────────────
k1 = make_cache_key('hello', 'world')
k2 = make_cache_key('hello', 'world')   # same → same key
k3 = make_cache_key('world', 'hello')   # different order → different key

print(f'make_cache_key("hello", "world") → {k1}')
print(f'make_cache_key("hello", "world") → {k2}  ← stable (identical)')
print(f'make_cache_key("world", "hello") → {k3}  ← different args = different key')
print()

# ── Dict arguments — order-independent ───────────────────────────────────────
kd1 = make_cache_key(params={'model': 'gpt-4', 'temperature': 0.7, 'max_tokens': 512})
kd2 = make_cache_key(params={'temperature': 0.7, 'max_tokens': 512, 'model': 'gpt-4'})
print(f'Dict arg (insertion order A) → {kd1}')
print(f'Dict arg (insertion order B) → {kd2}  ← same key (sort_keys=True)')
print()

# ── Nested structures ─────────────────────────────────────────────────────────
kn = make_cache_key('embed', ['token1', 'token2', 'token3'], version=2)
print(f'Nested list arg → {kn}')
print()

# ── Practical use: manual cache key for agent calls ──────────────────────────
agent_name = 'EmbeddingAgent'
inputs     = {'text': 'The quick brown fox', 'model': 'text-embedding-3-small'}
agent_key  = make_cache_key(agent_name, **inputs)
print(f'Agent cache key: {agent_key}')
print()

# ── Key length is always 16 hex chars ─────────────────────────────────────────
examples = [
    ('short string',),
    ('a' * 1000,),
    ({'deeply': {'nested': {'dict': [1, 2, 3]}}},),
]
print('Key length is always 16 chars:')
for args in examples:
    k = make_cache_key(*args)
    print(f'  args[0]={str(args[0])[:40]!r:<42}  key={k}  len={len(k)}')


## Section 7 — MultiTierCache: L1 + L2 Write-Through

`MultiTierCache` implements a **two-level cache hierarchy**:

```
Request
   ↓
L1 (small, fast, short TTL)
   ↓ miss
L2 (large, slower, longer TTL)
   ↓ miss
compute_fn (actual work)
```

**Write-through**: on every `set`, the value is written to both L1 and L2
simultaneously.

**L2 backfill on L2 hit**: if L1 misses but L2 hits, the value is automatically
copied back into L1 before being returned — so subsequent accesses find it in L1.

This mirrors CDN architectures: L1 is the edge cache (small, fast, volatile) and
L2 is the regional cache (large, slower, long-lived).


In [ ]:
# L1: small and short-lived — 4 entries, 5-second TTL
# L2: larger and longer-lived — 32 entries, 60-second TTL
l1 = AsyncCache(maxsize=4,  default_ttl=5.0)
l2 = AsyncCache(maxsize=32, default_ttl=60.0)

mtc = MultiTierCache(l1=l1, l2=l2)

# ── Write-through: set goes to both tiers ─────────────────────────────────────
await mtc.set('result:query_a', {'answer': 'Paris', 'confidence': 0.99})
await mtc.set('result:query_b', {'answer': 'London', 'confidence': 0.95})

l1_a = await l1.get('result:query_a')
l2_a = await l2.get('result:query_a')
print('After set("result:query_a"):')
print(f'  L1.get → {l1_a}')
print(f'  L2.get → {l2_a}')
print(f'  Both tiers contain the value (write-through confirmed).')
print()

# ── L1 hit path ───────────────────────────────────────────────────────────────
val = await mtc.get('result:query_a')
print(f'MultiTierCache.get("result:query_a") [L1 hit] → {val}')
print()

# ── L2 backfill on L2 hit ─────────────────────────────────────────────────────
# Manually evict from L1 to simulate an L1 miss
await l1.delete('result:query_a')
print('Manually deleted "result:query_a" from L1:')
print(f'  L1.get → {await l1.get("result:query_a")!r}  ← missing from L1')
print(f'  L2.get → {await l2.get("result:query_a")!r}  ← still in L2')
print()

# Now get via MultiTierCache — should trigger L2 backfill into L1
val_after = await mtc.get('result:query_a')
l1_after  = await l1.get('result:query_a')
print(f'MultiTierCache.get after L1 eviction [L2 hit + backfill]:')
print(f'  returned value    → {val_after}')
print(f'  L1 after backfill → {l1_after}  ← repopulated from L2')
print()

# ── Stats for both tiers ──────────────────────────────────────────────────────
stats = await mtc.stats()
print('MultiTierCache stats:')
print(f'  L1: {stats["l1"]}')
print(f'  L2: {stats["l2"]}')


## Section 8 — CacheManager: Named Registry

`CacheManager` maintains a **named dictionary of `AsyncCache` instances**.  You
register named caches at startup and then address them by name at call sites.

This is useful when different parts of a pipeline have different caching
requirements:

- `"embeddings"` cache: large, long TTL, high maxsize
- `"agent_responses"` cache: smaller, shorter TTL
- `"default"` cache: general-purpose fallback (auto-created)

### Key methods

| Method | Description |
|--------|-------------|
| `register(name, cache)` | Add a named `AsyncCache` |
| `get_cache(name)` | Return the named cache (auto-creates if missing) |
| `get(key, backend)` | Get a value from the named cache |
| `set(key, value, ttl, backend)` | Set a value in the named cache |
| `invalidate(key, backend)` | Delete a key from the named cache |
| `clear_all()` | Clear every registered cache |
| `stats()` | Return stats for all registered caches |


In [ ]:
mgr_cache = CacheManager()

# Register specialist caches
mgr_cache.register('embeddings',      AsyncCache(maxsize=2048, default_ttl=3600.0))  # 1-hour
mgr_cache.register('agent_responses', AsyncCache(maxsize=512,  default_ttl=120.0))   # 2-minute

# Store values in different backends
await mgr_cache.set('emb:hello',          [0.1, 0.2, 0.3],            backend='embeddings')
await mgr_cache.set('emb:world',          [0.4, 0.5, 0.6],            backend='embeddings')
await mgr_cache.set('resp:translate_en',  'Hello',                     backend='agent_responses')
await mgr_cache.set('resp:translate_es',  'Hola',                      backend='agent_responses')
await mgr_cache.set('general_setting',    {'timeout': 30},             backend='default')

# Retrieve from specific backends
print('Reads from named backends:')
print(f'  embeddings["emb:hello"]           → {await mgr_cache.get("emb:hello",         backend="embeddings")}')
print(f'  agent_responses["resp:translate_es"] → {await mgr_cache.get("resp:translate_es", backend="agent_responses")!r}')
print(f'  default["general_setting"]        → {await mgr_cache.get("general_setting",    backend="default")}')
print()

# Invalidate a single key
removed = await mgr_cache.invalidate('resp:translate_en', backend='agent_responses')
print(f'invalidate("resp:translate_en", backend="agent_responses") → {removed}')
print(f'  get after invalidate → {await mgr_cache.get("resp:translate_en", backend="agent_responses")!r}')
print()

# Stats across all named caches
all_stats = await mgr_cache.stats()
print('CacheManager stats (all backends):')
for name, s in all_stats.items():
    print(f'  {name:<18}: {s}')
print()

# clear_all empties every registered cache
await mgr_cache.clear_all()
print('After clear_all():')
print(f'  embeddings["emb:hello"]           → {await mgr_cache.get("emb:hello", backend="embeddings")!r}')
print(f'  default["general_setting"]        → {await mgr_cache.get("general_setting", backend="default")!r}')


## Section 9 — Cache with Expensive Computation: Measuring Speedup

This section simulates a realistic scenario: an agent function that calls an
external API (modelled as `asyncio.sleep`) and is expensive to run.  We measure
wall-clock time with and without caching to quantify the benefit.


In [ ]:
import random

_slow_call_count = 0

async def slow_agent_fn(topic: str, depth: int = 2) -> dict:
    """
    Simulate an agent that calls an LLM API.
    Takes ~100–150 ms per call.
    """
    global _slow_call_count
    _slow_call_count += 1
    delay = 0.10 + random.uniform(0, 0.05)   # 100–150 ms
    await asyncio.sleep(delay)
    return {
        'topic':    topic,
        'depth':    depth,
        'response': f'Analysis of {topic!r} at depth {depth}: key finding #{_slow_call_count}',
        'tokens':   len(topic.split()) * 12,
    }

# ── Without caching ────────────────────────────────────────────────────────────
_slow_call_count = 0
topics = ['AI safety', 'climate change', 'AI safety', 'quantum computing', 'climate change']

t_start = time.perf_counter()
uncached_results = []
for topic in topics:
    result = await slow_agent_fn(topic)
    uncached_results.append(result)
uncached_time = time.perf_counter() - t_start

print(f'Without caching ({len(topics)} calls, {_slow_call_count} unique):')
print(f'  Total time      : {uncached_time*1000:.0f} ms')
print(f'  Actual API calls: {_slow_call_count}')
print()

# ── With caching ───────────────────────────────────────────────────────────────
_slow_call_count = 0

@cached(ttl=60.0, maxsize=64)
async def cached_agent_fn(topic: str, depth: int = 2) -> dict:
    return await slow_agent_fn(topic, depth)

t_start = time.perf_counter()
cached_results = []
for topic in topics:
    result = await cached_agent_fn(topic)
    cached_results.append(result)
cached_time = time.perf_counter() - t_start

print(f'With @cached ({len(topics)} calls, {_slow_call_count} actual computations):')
print(f'  Total time      : {cached_time*1000:.0f} ms')
print(f'  Actual API calls: {_slow_call_count}  ({len(topics) - _slow_call_count} cache hits)')
print()

speedup = uncached_time / cached_time if cached_time > 0 else float('inf')
print(f'Speedup: {speedup:.1f}x  (total time {uncached_time*1000:.0f} ms → {cached_time*1000:.0f} ms)')
print()

# Verify result consistency
print('Result consistency check (cached == uncached for matching topics):')
topic_to_uncached = {r['topic']: r['response'] for r in uncached_results}
topic_to_cached   = {r['topic']: r['response'] for r in cached_results}
for topic in set(topics):
    # Topics computed on the same run are consistent; we just check keys present
    print(f'  topic={topic!r:<20}  in both results: {topic in topic_to_uncached and topic in topic_to_cached}')


## Section 10 — Cache Invalidation Patterns

Cache invalidation is notoriously one of the two hard problems in computer science.
The Multigen caching system supports three invalidation strategies:

| Strategy | How | When to use |
|----------|-----|-------------|
| **Manual / explicit** | `cache.delete(key)` or `cache.clear()` | After a data mutation (write-through invalidation) |
| **TTL-based** | Set `ttl=` on entry; expires automatically | Time-sensitive data (tokens, prices) |
| **LRU capacity-based** | Exceeding `maxsize` evicts LRU entry | Passive — happens automatically |

This section demonstrates all three patterns with worked examples.


In [ ]:
inv_cache = AsyncCache(maxsize=10, default_ttl=60.0)

# ── Pattern 1: Manual explicit invalidation ────────────────────────────────────
await inv_cache.set('user:alice', {'name': 'Alice', 'role': 'admin'})
await inv_cache.set('user:bob',   {'name': 'Bob',   'role': 'viewer'})

print('=== Pattern 1: Manual invalidation ===')
print(f'  Before: user:alice → {await inv_cache.get("user:alice")}')

# Simulate a write operation that invalidates the cache entry
# (e.g., user role was updated in the database)
await inv_cache.delete('user:alice')
print(f'  After delete: user:alice → {await inv_cache.get("user:alice")!r}  ← invalidated')
print()

# ── Pattern 2: TTL-based invalidation ─────────────────────────────────────────
print('=== Pattern 2: TTL-based expiry ===')
await inv_cache.set('rate_limit:user:bob', {'remaining': 95, 'window': '1m'}, ttl=0.1)  # 100 ms
print(f'  Immediately: rate_limit:user:bob → {await inv_cache.get("rate_limit:user:bob")}')
time.sleep(0.15)
print(f'  After 150 ms: rate_limit:user:bob → {await inv_cache.get("rate_limit:user:bob")!r}  ← expired')
print()

# ── Pattern 3: LRU capacity-based eviction ────────────────────────────────────
print('=== Pattern 3: LRU capacity eviction (maxsize=4) ===')
lru_inv = AsyncCache(maxsize=4)

await lru_inv.set('k1', 'v1')
await lru_inv.set('k2', 'v2')
await lru_inv.set('k3', 'v3')
await lru_inv.set('k4', 'v4')

# Access k1, k2, k3 to make k4 the LRU
_ = await lru_inv.get('k1')
_ = await lru_inv.get('k2')
_ = await lru_inv.get('k3')

# Insert k5 — k4 (LRU) is evicted
await lru_inv.set('k5', 'v5')
stats = await lru_inv.stats()
print(f'  After inserting 5th entry (maxsize=4): size={stats["size"]}')
print(f'  k4 (LRU) evicted: get("k4") → {await lru_inv.get("k4")!r}')
print(f'  k5 (MRU) present: get("k5") → {await lru_inv.get("k5")!r}')
print(f'  Stats: {stats}')
print()

# ── Pattern 4: Tag-based / prefix-based bulk invalidation ─────────────────────
print('=== Pattern 4: Prefix-based bulk invalidation ===')
# AsyncCache doesn't have native tag support, but a prefix pattern is easy to
# implement: store keys in a set, then delete by prefix.

prefix_cache = AsyncCache(maxsize=100)
keys_by_prefix: dict = {'user:': [], 'session:': []}

async def set_tracked(key: str, value):
    await prefix_cache.set(key, value)
    for prefix in keys_by_prefix:
        if key.startswith(prefix):
            keys_by_prefix[prefix].append(key)

async def invalidate_prefix(prefix: str):
    for key in keys_by_prefix.pop(prefix, []):
        await prefix_cache.delete(key)

await set_tracked('user:alice', {'name': 'Alice'})
await set_tracked('user:bob',   {'name': 'Bob'})
await set_tracked('session:s1', {'token': 'tok-abc'})

print(f'  Before bulk invalidate: user:alice={await prefix_cache.get("user:alice")}')
await invalidate_prefix('user:')
print(f'  After invalidate_prefix("user:"): user:alice={await prefix_cache.get("user:alice")!r}')
print(f'  session:s1 untouched: {await prefix_cache.get("session:s1")}')


## Section 11 — Caching in an Agent Pipeline

This section shows how `@cached` integrates into a real agent pipeline built with
`FunctionAgent`.  We wrap an expensive classification agent with the decorator and
observe that repeated calls with the same arguments are served from cache.


In [ ]:
from multigen import FunctionAgent, Chain

pipeline_call_log = []

# ── Define a cached agent function ────────────────────────────────────────────

@cached(ttl=120.0, maxsize=512)
async def classify_text(text: str, categories: tuple) -> dict:
    """
    Simulate an expensive text classification LLM call.
    We use a tuple for categories so it is hashable and cache-key-stable.
    """
    pipeline_call_log.append({'text': text, 'categories': categories})
    await asyncio.sleep(0.04)   # 40 ms "model inference"
    scores = {cat: round(0.3 + hash(f'{text}{cat}') % 100 / 200, 3) for cat in categories}
    best = max(scores, key=scores.__getitem__)
    return {'text': text, 'scores': scores, 'label': best, 'cached': False}

# ── Wrap in FunctionAgent ──────────────────────────────────────────────────────

async def classification_agent_fn(ctx: dict) -> dict:
    text = ctx.get('input_text', '')
    cats = tuple(ctx.get('categories', ['tech', 'finance', 'health']))
    result = await classify_text(text, cats)
    return {'classification': result, 'text': text}

classify_agent = FunctionAgent('ClassifyAgent', fn=classification_agent_fn)

# ── Preprocessing and postprocessing agents ────────────────────────────────────

preprocess_agent = FunctionAgent(
    'PreprocessAgent',
    fn=lambda ctx: {
        'input_text': ctx.get('raw_text', '').strip().lower(),
        'categories': ctx.get('categories', ['tech', 'finance', 'health']),
    }
)

postprocess_agent = FunctionAgent(
    'PostprocessAgent',
    fn=lambda ctx: {
        'final_label':     ctx.get('classification', {}).get('label', 'unknown'),
        'confidence':      max(ctx.get('classification', {}).get('scores', {0: 0}).values(), default=0),
        'raw_text':        ctx.get('raw_text', ''),
    }
)

# ── Build the pipeline ────────────────────────────────────────────────────────

pipeline = Chain([preprocess_agent, classify_agent, postprocess_agent])

# ── Run with repeated inputs ───────────────────────────────────────────────────
requests = [
    {'raw_text': 'Federal Reserve raises interest rates again', 'categories': ['tech', 'finance', 'health']},
    {'raw_text': 'New AI model beats GPT-4 on benchmarks',      'categories': ['tech', 'finance', 'health']},
    {'raw_text': 'Federal Reserve raises interest rates again', 'categories': ['tech', 'finance', 'health']},  # duplicate
    {'raw_text': 'Cancer treatment breakthrough announced',     'categories': ['tech', 'finance', 'health']},
    {'raw_text': 'New AI model beats GPT-4 on benchmarks',      'categories': ['tech', 'finance', 'health']},  # duplicate
]

pipeline_call_log.clear()
t0 = time.perf_counter()

results = []
for req in requests:
    out = await pipeline.run(req)
    results.append(out)

pipeline_time = time.perf_counter() - t0

print(f'Pipeline ran {len(requests)} requests in {pipeline_time*1000:.0f} ms')
print(f'Actual classify_text calls: {len(pipeline_call_log)} (expected 3 — 2 were cache hits)')
print()
print('Per-request results:')
for i, (req, res) in enumerate(zip(requests, results)):
    print(f'  [{i}] {req["raw_text"][:45]!r:<47} → label={res["final_label"]!r}')
print()
cache_stats = await classify_text.cache.stats()
print(f'@cached stats: {cache_stats}')


## Section 12 — Summary: When to Use Each Cache Tier

### Tier selection guide

| Tier | Class | Async | Eviction | TTL | Best for |
|------|-------|-------|----------|-----|----------|
| LRU (sync) | `LRUCache` | No | LRU | Optional | CPU-bound sync computations, small hot-sets |
| TTL (sync) | `TTLCache` | No | TTL + oldest-on-full | Required | Auth tokens, rate-limit buckets, OTP codes |
| Async | `AsyncCache` | Yes | LRU + TTL | Optional | LLM calls, embedding APIs, any async I/O |
| Multi-tier | `MultiTierCache` | Yes | L1 LRU + L2 LRU | Per-tier | Hot/warm access patterns, CDN-style caching |

### Configuration guide

```python
# 1. Cheap, sync, in-process lookup (e.g., parsed config)
cache = LRUCache(maxsize=1024, default_ttl=None)

# 2. Time-limited tokens (e.g., OAuth bearer token, 1-hour window)
tokens = TTLCache(default_ttl=3600.0, maxsize=512)

# 3. Async LLM result cache with stampede protection
llm_cache = AsyncCache(maxsize=2048, default_ttl=3600.0)

# 4. Two-tier hot/warm caching
cache = MultiTierCache(
    l1=AsyncCache(maxsize=64,   default_ttl=30.0),   # hot: 30 s
    l2=AsyncCache(maxsize=2048, default_ttl=3600.0), # warm: 1 h
)

# 5. Decorate an expensive async function
@cached(ttl=600.0, maxsize=256)
async def embed(text: str) -> list: ...

# 6. Manage multiple named caches in a service
mgr = CacheManager()
mgr.register("embeddings",  AsyncCache(maxsize=4096, default_ttl=86400.0))
mgr.register("llm_results", AsyncCache(maxsize=1024, default_ttl=3600.0))
```

### Invalidation cheat-sheet

| Pattern | Code |
|---------|------|
| Invalidate single key | `await cache.delete(key)` |
| Invalidate all entries | `await cache.clear()` |
| Auto-expire after time | `cache.set(key, val, ttl=N)` |
| Auto-evict when full | LRU eviction on insert (automatic) |
| Clear all named caches | `await mgr.clear_all()` |

### Key generation

```python
# Recommended: use make_cache_key for complex argument types
key = make_cache_key(agent_name, query, model=model_name)

# Custom key function for @cached
@cached(ttl=60, key_fn=lambda text, model: f"embed:{model}:{hash(text)}")
async def embed(text: str, model: str) -> list: ...
```

### Next steps

- **Notebook 21**: Multi-Tier Memory System — ShortTermMemory, EpisodicMemory,
  WorkingMemory, SemanticMemory, MemoryManager
- Combine `MultiTierCache` with `MemoryManager` for a complete context-management
  and caching stack.
- Replace `asyncio.sleep` mocks with real embedding or LLM client calls to see
  real-world cache hit rates.
